In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## Dataset Loading
- ESI2 uses 2 Datasets hence as it uses 2/3 of the data

In [ ]:
from datasets import load_dataset

dataset1 = load_dataset("csv", data_files="/content/ES2Data.csv", split="train")
dataset2= load_dataset("csv", data_files="/content/ES3Data.csv", split="train")


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from datasets import concatenate_datasets

dataset = concatenate_datasets([dataset1, dataset2])
print(dataset)

Dataset({
    features: ['Unnamed: 0', 'instruction', 'input', 'output'],
    num_rows: 6628
})


## Mount to Drive

- Mount to Drive for reproducability

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/colab_runs/es2_agent_final"

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/colab_runs/es2_agent_final"
os.makedirs(BASE_DIR, exist_ok=True)

CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
LOG_DIR = os.path.join(BASE_DIR, "logs")
MODEL_DIR = os.path.join(BASE_DIR, "final_model")

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## Model Training

- Train the Model using PEFT and with LoRA
- Use Unsloth, TRL and LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/medgemma-4b-it",
    max_seq_length = 2048,
    dtype= None,
    load_in_4bit= True

)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
def format_example(example):
    instruction = str(example["instruction"] or "").strip()
    user_input = str(example["input"] or "").strip()
    output = str(example["output"] or "").strip()

    messages = [
        {
            "role": "user",
            "content": f"{instruction}\n\nPatient case:\n{user_input}"
        },
        {
            "role": "assistant",
            "content": output
        },
    ]
    return {"messages": messages}


def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_dataset = dataset.map(format_example)
formatted_dataset = formatted_dataset.map(apply_template)

Map:   0%|          | 0/6628 [00:00<?, ? examples/s]

Map:   0%|          | 0/6628 [00:00<?, ? examples/s]

In [ ]:
max_len = 1024

lengths = [
    len(tokenizer(x["text"], add_special_tokens=True)["input_ids"][0])
    for x in formatted_dataset
]

truncated = sum(1 for l in lengths if l > max_len)

print("Average length:", sum(lengths) / len(lengths))
print("Max length:", max(lengths))
print("Examples over", max_len, "tokens:", truncated)
print("Percent truncated:", 100 * truncated / len(lengths), "%")

Average length: 486.8821665660833
Max length: 2118
Examples over 1024 tokens: 25
Percent truncated: 0.3771876885938443 %


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing = "unsloth"

)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [ ]:
import os

checkpoint_DIR = os.path.join(BASE_DIR, "checkpoints", "checkpoint-1100")

print(os.listdir(checkpoint_DIR))
print(checkpoint_DIR)

['tokenizer.json', 'adapter_config.json', 'README.md', 'scheduler.pt', 'processor_config.json', 'tokenizer.model', 'optimizer.pt', 'chat_template.jinja', 'training_args.bin', 'tokenizer_config.json', 'trainer_state.json', 'adapter_model.safetensors', 'rng_state.pth']
/content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1100


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=1024,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=5e-5,
        warmup_steps=10,
        logging_steps=10,
        output_dir=CKPT_DIR,
        optim="adamw_8bit",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
        packing=False,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/6628 [00:00<?, ? examples/s]

In [ ]:
trainer.train(
    resume_from_checkpoint=checkpoint_DIR,
)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,628 | Num Epochs = 2 | Total steps = 1,658
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 32,788,480 of 4,332,867,952 (0.76% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1110,0.511642
1120,0.509037
1130,0.492892
1140,0.477325
1150,0.492529
1160,0.518063
1170,0.507181
1180,0.507683
1190,0.500494
1200,0.473756


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1200/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1200.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1300.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1400/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1400.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/checkpoint-1500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/colab_runs/es2_agent_final/checkpoints/check

TrainOutput(global_step=1658, training_loss=0.16648597869711992, metrics={'train_runtime': 6873.4812, 'train_samples_per_second': 1.929, 'train_steps_per_second': 0.241, 'total_flos': 1.684641701573376e+17, 'train_loss': 0.16648597869711992, 'epoch': 2.0})

In [ ]:
from unsloth import FastLanguageModel
import torch

CHECKPOINT_PATH = "/content/drive/MyDrive/colab_runs/es1_agent_v2/checkpoints/checkpoint-752"
max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT_PATH,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.11: Fast Gemma3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForConditionalGeneration(
      (model): Gemma3Model(
        (vision_tower): SiglipVisionModel(
          (vision_model): SiglipVisionTransformer(
            (embeddings): SiglipVisionEmbeddings(
              (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
              (position_embedding): Embedding(4096, 1152)
            )
            (encoder): SiglipEncoder(
              (layers): ModuleList(
                (0-15): 16 x SiglipEncoderLayer(
                  (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
                  (self_attn): SiglipAttention(
                    (k_proj): lora.Linear(
                      (base_layer): Linear(in_features=1152, out_features=1152, bias=True)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.05, inplace=False)
                      )
                      (lor

In [ ]:
prompt = """### Instruction:
Analyze the patient’s condition and assess if they require immediate life-saving interventions. Focus on identifying critical needs in airway, breathing, circulation, or severe physiological instability. Provide specific reasons for your assessment.

### Input:
A 70-year-old male with stage IV pancreatic cancer presents with dyspnea, dizziness, diarrhea, hypotension (BP 86/57), pulse 145, RR 30, SpO2 100%, poor oral intake, and dehydration.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Analyze the patient’s condition and assess if they require immediate life-saving interventions. Focus on identifying critical needs in airway, breathing, circulation, or severe physiological instability. Provide specific reasons for your assessment.

### Input:
A 70-year-old male with stage IV pancreatic cancer presents with dyspnea, dizziness, diarrhea, hypotension (BP 86/57), pulse 145, RR 30, SpO2 100%, poor oral intake, and dehydration.

### Response:
The patient is a 70-year-old male with stage IV pancreatic cancer, presenting with several concerning symptoms that suggest he may be experiencing a critical condition. The symptoms include dyspnea (difficulty breathing), dizziness, diarrhea, hypotension (low blood pressure), tachycardia (high heart rate), tachypnea (high respiratory rate), and poor oral intake with dehydration.

### Analysis:
1. **Hypotension and Tachycardia**: The combination of low blood pressure and high heart rate indicates possible shock, which 